# SelectStream-Lite: Ablations P2, P3, and P4 on Google Colab

This notebook runs the pending ablations described in `ablations_for_team.md`:
- **P2: Standard Metrics & Fine-Grained Breakdowns** (CPU, ~1–2 min, 0 CU)
- **P3: Plain Qwen Oracle & Uniform Frame Baselines** (A100 GPU, ~50 min, ~11–13 CU)
- **P4: Multi-Seed Stability (Seeds 43 & 44, Arms A & B)** (A100 GPU, ~6.5–7.5 h, ~85–95 CU)

### 🛡️ Safety & Reliability Design:
1. **Read-Only Data:** The video clips directory (`ego4d_nlq/clips`) in your Google Drive is strictly read-only.
2. **Safe Workspace:** All code, outputs, predictions, logs, and checkpoints reside in `/content/drive/MyDrive/selectstream_lite`.
3. **Internet Disconnect Resilient:** Long GPU runs (P3 and P4) run asynchronously in the background via `nohup`. You can close your laptop or lose Wi-Fi connection and the jobs will continue running on Colab. Checkpoints auto-save every 10 streams so jobs can resume if preempted.
4. **Budget Management:** Total GPU usage is ~95 to 110 Compute Units, fitting safely within your 200 CU budget.

## Step 0: Colab Environment Setup & Mount Google Drive

In [5]:
# 1. Check GPU
!nvidia-smi

# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os, sys
DRIVE_ROOT = "/content/drive/MyDrive"
REPO_DIR = os.path.join(DRIVE_ROOT, "selectstream_lite")
VIDEO_DIR = os.path.join(DRIVE_ROOT, "ego4d_nlq/clips")
WORK_DIR = REPO_DIR  # All outputs and checkpoints stay in selectstream_lite

# Verify paths
assert os.path.isdir(REPO_DIR), f"Repository directory not found at {REPO_DIR}"
assert os.path.isdir(VIDEO_DIR), f"Video directory not found at {VIDEO_DIR}"

# Set working directory directly to selectstream_lite on Drive
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ["PYTHONPATH"] = f"{REPO_DIR}:" + os.environ.get("PYTHONPATH", "")

# Verify 'main' module imports cleanly
import main
print(f"✅ Working directory set to: {os.getcwd()}")
print(f"✅ Successfully imported 'main' module from: {main.__file__}")

clip_count = len([f for f in os.listdir(VIDEO_DIR) if f.endswith('.mp4')])
print(f"✅ Video directory verified (READ-ONLY): {VIDEO_DIR} ({clip_count} clips found)")

# Create working folders
os.makedirs(os.path.join(WORK_DIR, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(WORK_DIR, "eval"), exist_ok=True)
os.makedirs(os.path.join(WORK_DIR, "logs"), exist_ok=True)
print(f"✅ Workspace ready: {WORK_DIR}")

Thu Oct 29 15:02:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             49W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Step 1: Install Dependencies & Download Checkpoints from Hugging Face

In [6]:
# Install packages
!pip install -q "transformers>=4.45.0,<5.6.0" "huggingface_hub<2.0.0" torchvision opencv-python-headless

from huggingface_hub import login, snapshot_download

# Log in with your Hugging Face read access token
login("HF_TOKEN")

# 1. Download checkpoints, scorer, calibration, and existing predictions (~3 GB)
print("Downloading checkpoints and existing predictions...")
snapshot_download(
    repo_id="amritha27/selectstream-lite-ego4d",
    local_dir="/content/ck",
    allow_patterns=[
        "checkpoints/full_v3/R_A/*",
        "runs/full_v3/scorer/*",
        "runs/full_v3/calibration.json",
        "eval/*",
        "runs/full_v3/*"
    ]
)

# 2. Download feature cache g/ and meta/ (~12 GB)
print("Downloading feature cache...")
snapshot_download(
    repo_id="amritha27/selectstream-lite-ego4d-features",
    repo_type="dataset",
    local_dir="/content/feature_cache"
)
print("✅ Downloads complete!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 202 files:   0%|          | 0/202 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3374 files:   0%|          | 0/3374 [00:00<?, ?it/s]

✅ Downloads complete!


--- 
## Step 2: Run P2 (Standard Metrics & Breakdowns from Existing Predictions)
* **Compute:** CPU only (0 GPU Compute Units)
* **Runtime:** ~1 to 2 minutes
* **What it does:** Parses existing prediction files (`predictions.jsonl`) for QA and NLQ tracks, computes NLQ-style R@1 / R@5 at IoU $\ge 0.3$ and $0.5$, token F1, lenient matches, breakdowns by delay bucket/disjoint participants, and paired bootstrap statistics.

In [7]:
import glob, os, sys, subprocess

# Ensure environment has selectstream_lite in PYTHONPATH
env = dict(os.environ, PYTHONPATH=REPO_DIR)

preds = sorted(glob.glob("/content/ck/**/predictions.jsonl", recursive=True))
print(f"Found {len(preds)} prediction files from Hub:")
for p in preds:
    print(" -", p)

# Separate QA and NLQ tracks for clean, meaningful comparison
qa_preds = [p for p in preds if "qa" in p]
nlq_preds = [p for p in preds if "nlq" in p]

if qa_preds:
    print("\n===========================================================")
    print("📊 Running P2 Breakdown on QA Track Predictions")
    print("===========================================================")
    qa_args = [f"{os.path.basename(os.path.dirname(p))}={p}" for p in qa_preds]
    out_qa = os.path.join(WORK_DIR, "eval/breakdown_qa")
    cmd_qa = [
        sys.executable, "-m", "main.cli.eval_metrics_breakdown",
        "--preds", *qa_args,
        "--ref", "eval_qa_A",
        "--out_dir", out_qa
    ]
    subprocess.run(cmd_qa, env=env, check=True)

if nlq_preds:
    print("\n===========================================================")
    print("📊 Running P2 Breakdown on NLQ Track Predictions")
    print("===========================================================")
    nlq_args = [f"{os.path.basename(os.path.dirname(p))}={p}" for p in nlq_preds]
    out_nlq = os.path.join(WORK_DIR, "eval/breakdown_nlq")
    cmd_nlq = [
        sys.executable, "-m", "main.cli.eval_metrics_breakdown",
        "--preds", *nlq_args,
        "--ref", "eval_nlq_A",
        "--out_dir", out_nlq
    ]
    subprocess.run(cmd_nlq, env=env, check=True)

Found 7 prediction files from Hub:
 - /content/ck/runs/full_v3/eval_nlq_A/predictions.jsonl
 - /content/ck/runs/full_v3/eval_nlq_B/predictions.jsonl
 - /content/ck/runs/full_v3/eval_nlq_C/predictions.jsonl
 - /content/ck/runs/full_v3/eval_qa_A/predictions.jsonl
 - /content/ck/runs/full_v3/eval_qa_B/predictions.jsonl
 - /content/ck/runs/full_v3/eval_qa_C/predictions.jsonl
 - /content/ck/runs/full_v3/eval_qa_untrained/predictions.jsonl

📊 Running P2 Breakdown on QA Track Predictions

📊 Running P2 Breakdown on NLQ Track Predictions


--- 
## Step 3: Run P3 (Plain Qwen Oracle & Uniform Frames)
* **Compute:** A100 GPU
* **Runtime:** ~50 minutes total (~25 min for Oracle frames, ~25 min for Uniform frames)
* **Compute Units:** ~**11 to 13 CU**
* **Resilience:** Launched asynchronously in background with `nohup`. Video clips are accessed in read-only mode.

In [563]:
!pkill -f "colab_run_p3.sh"
!pkill -f "main.cli.eval_ego4d"

In [564]:
!ps aux | grep eval_ego4d

root      145306  0.0  0.0   7348  3600 ?        S    22:19   0:00 /bin/bash -c ps aux | grep eval_ego4d
root      145308  0.0  0.0   6552  2420 ?        S    22:19   0:00 grep eval_ego4d


In [359]:
import os, shutil, json
from collections import defaultdict

DRIVE_CLIPS_DIR = "/content/drive/MyDrive/ego4d_nlq/clips"
LOCAL_CLIPS_DIR = "/content/clips"
os.makedirs(LOCAL_CLIPS_DIR, exist_ok=True)

# Find the clips needed for the first 50 validation streams
clean_clips = json.load(open("dataset/clean/clips.json"))
by_vid = defaultdict(list)
for c in clean_clips:
    if c.get("split") == "val":
        by_vid[c["video_uid"]].append(c["clip_uid"])

# Gather clip UIDs for the first 50 streams
val_vids = sorted(by_vid.keys())[:50]
target_uids = {u for v in val_vids for u in by_vid[v]}
print(f"📦 Found {len(target_uids)} clips needed for the 50 validation streams.")

# Copy clips to local SSD with progress
copied = 0
for uid in target_uids:
    src = os.path.join(DRIVE_CLIPS_DIR, f"{uid}.mp4")
    dst = os.path.join(LOCAL_CLIPS_DIR, f"{uid}.mp4")
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copyfile(src, dst)
        copied += 1

print(f"⚡ Successfully copied {copied} clips to local SSD ({LOCAL_CLIPS_DIR})!")
print(f"Local clips count: {len(os.listdir(LOCAL_CLIPS_DIR))}")

📦 Found 79 clips needed for the 50 validation streams.
⚡ Successfully copied 0 clips to local SSD (/content/clips)!
Local clips count: 80


In [297]:
!cp /content/drive/MyDrive/ego4d_nlq/clips/f3e4cdf4-73fa-489a-8be3-c9265364da52.mp4 /content/clips/ 2>/dev/null || true

In [360]:
# Launch P3 in background (Internet-disconnect resilient)
!nohup bash scripts/colab_run_p3.sh >> /content/drive/MyDrive/selectstream_lite/logs/p3.log 2>&1 &
print("🚀 P3 started in the background!")
print("Logging to: /content/drive/MyDrive/selectstream_lite/logs/p3.log")

🚀 P3 started in the background!
Logging to: /content/drive/MyDrive/selectstream_lite/logs/p3.log


In [708]:
# Monitor P3 progress in real-time (Run this cell anytime to see latest status)
import time
log_file = "/content/drive/MyDrive/selectstream_lite/logs/p3.log"

if os.path.exists(log_file):
    print(f"--- Status at {time.strftime('%X')} ---")
    !tail -n 15 {log_file}
else:
    print("Waiting for log file to be created...")

--- Status at 08:00:07 ---
}
[P3 Runner] ===== P3 oracle done at Thu Oct 29 09:56:49 PM UTC 2026 =====
[P3 Runner] Running Uniform Frames baseline (max_clips=50, resume=enabled) ...
[arm][WARN] A: no calibration file (outputs/calibration.json); using config values {}
[env] GPUs: ['NVIDIA A100-SXM4-80GB']
Loading weights: 100%|██████████| 729/729 [00:05<00:00, 145.10it/s]
/content/drive/MyDrive/selectstream_lite/main/model/query_builder.py:26: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
/content/drive/MyDrive/selectstream_lite/main/model/lvm_memory.py:264: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.segment_encoder = nn.TransformerEncoder(seg_encoder_layer, num_layers=self.segment_encoder_layers)
[eval] feature cache /content/feature_cache: 243/243 stre

In [709]:
# Check how many questions have been answered so far
!wc -l /content/drive/MyDrive/selectstream_lite/eval/*/predictions.jsonl 2>/dev/null

    867 /content/drive/MyDrive/selectstream_lite/eval/p3_oracle_frames/predictions.jsonl
      0 /content/drive/MyDrive/selectstream_lite/eval/p3_uniform_frames/predictions.jsonl
    899 /content/drive/MyDrive/selectstream_lite/eval/W_A_seed43_nlq/predictions.jsonl
    867 /content/drive/MyDrive/selectstream_lite/eval/W_A_seed43_qa/predictions.jsonl
    899 /content/drive/MyDrive/selectstream_lite/eval/W_B_seed43_nlq/predictions.jsonl
    867 /content/drive/MyDrive/selectstream_lite/eval/W_B_seed43_qa/predictions.jsonl
   4399 total


In [292]:
!nvidia-smi --query-gpu=utilization.gpu,utilization.memory,power.draw,memory.used --format=csv

utilization.gpu [%], utilization.memory [%], power.draw [W], memory.used [MiB]
0 %, 0 %, 70.50 W, 18748 MiB


In [565]:
import time, os, subprocess, glob

CHECK_INTERVAL_SEC = 180   # Local GPU check every 3 minutes
DRIVE_CHECK_EVERY = 10     # Check Drive every 10 cycles (10 * 3 min = 30 minutes)
LOG_FILE = "/content/drive/MyDrive/selectstream_lite/logs/p3.log"
EVAL_DIR = "/content/drive/MyDrive/selectstream_lite/eval"

print(f"🟢 Resilient Monitor started at {time.strftime('%X')}.")
print("• Local GPU checked every 3 min.")
print("• Questions answered & Drive log checked every 30 min.")
print("• Alarm set for ~03:00 AM.\n")

cycle = 0

while True:
    cycle += 1
    now = time.strftime('%X')

    # 1. Local GPU hardware check (zero Drive access)
    gpu_stat = subprocess.run(
        ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"],
        capture_output=True, text=True
    ).stdout.strip()

    # 2. Check if process is still running in memory
    res = subprocess.run(["pgrep", "-f", "main.cli"], capture_output=True, text=True)
    is_running = bool(res.stdout.strip())

    print(f"[{now}] GPU: {gpu_stat} | Process Active: {is_running}")

    # 3. Check Google Drive once every 30 minutes (Questions count + Log)
    if cycle % DRIVE_CHECK_EVERY == 0:
        try:
            # A. Count questions answered so far across predictions files
            pred_files = glob.glob(f"{EVAL_DIR}/p3_*/predictions.jsonl")
            q_counts = {}
            for pf in pred_files:
                tag = os.path.basename(os.path.dirname(pf))
                res_wc = subprocess.run(["wc", "-l", pf], capture_output=True, text=True, timeout=10)
                if res_wc.returncode == 0 and res_wc.stdout.strip():
                    count = res_wc.stdout.strip().split()[0]
                    q_counts[tag] = count

            q_summary = ", ".join([f"{k}: {v} questions answered" for k, v in q_counts.items()]) if q_counts else "Awaiting first buffer flush"
            print(f"  📊 [30-min Progress]: {q_summary}")

            # B. Read latest log line
            if os.path.exists(LOG_FILE):
                tail = subprocess.run(["tail", "-n", "3", LOG_FILE], capture_output=True, text=True, timeout=10).stdout.strip()
                if tail:
                    print(f"  📁 [Drive Log]: {tail.splitlines()[-1]}")

        except Exception as e:
            # Fallback: Drive latency or timeout will NEVER crash the monitor
            print(f"  ⚠️ [Drive Warning]: Temporary Drive read issue ({e}). Skipping without crashing.")

    # 4. Exit automatically when the job completes
    if not is_running:
        print(f"\n🎉 P3 Finished successfully at {time.strftime('%X')}! Time to start P4.")
        break

    time.sleep(CHECK_INTERVAL_SEC)

🟢 Resilient Monitor started at 22:19:25.
• Local GPU checked every 3 min.
• Questions answered & Drive log checked every 30 min.
• Alarm set for ~03:00 AM.

[22:19:25] GPU: 0 %, 0 MiB | Process Active: False

🎉 P3 Finished successfully at 22:19:25! Time to start P4.


--- 
## Step 4: Run P4 (Multi-Seed Training: Seeds 43 & 44, Arms A & B)
* **Compute:** A100 GPU
* **Runtime:** ~6.5 to 7.5 hours (4 Stage-W training runs $\times$ 50 steps + QA & NLQ evaluations)
* **Compute Units:** ~**85 to 95 CU**
* **Resilience:** Auto-saves `latest/` checkpoint every 10 streams on Google Drive. If disconnected, re-launching automatically resumes from the last completed checkpoint.

In [659]:
# Launch P4 in background (Internet-disconnect resilient)
!nohup bash scripts/colab_run_p4.sh >> /content/drive/MyDrive/selectstream_lite/logs/p4.log 2>&1 &
print("🚀 P4 multi-seed pipeline started in the background!")
print("Logging to: /content/drive/MyDrive/selectstream_lite/logs/p4.log")

🚀 P4 multi-seed pipeline started in the background!
Logging to: /content/drive/MyDrive/selectstream_lite/logs/p4.log


In [705]:
# Monitor P4 progress in real-time (Run this cell anytime to see latest status)
import time
log_file = "/content/drive/MyDrive/selectstream_lite/logs/p4.log"

if os.path.exists(log_file):
    print(f"--- Status at {time.strftime('%X')} ---")
    !tail -n 15 {log_file}
else:
    print("Waiting for log file to be created...")

--- Status at 05:19:30 ---
[ckpt] loaded /content/drive/MyDrive/selectstream_lite/checkpoints/W_A_seed43/final/weights.pt: 65 tensors, missing(non-backbone)=0, unexpected=0
[scorer] loaded /content/ck/runs/full_v3/scorer/scorer.pt (seed 42, dev AP 0.3148258249050078)
[eval] feature cache /content/feature_cache: 243/243 streams fully cached
[eval] arm=A track=qa split=val streams=50 missing_video=0
[eval] stream 1/50 000cd456-ff8d-499b-b0c1-4acead128a8b clips=1 frames=480 queries=4 34s
[eval] stream 2/50 0066ab25-04ad-41b2-89ab-283d2bfa1c4b clips=1 frames=480 queries=13 34s
[eval] stream 3/50 03e90bbc-7d6b-423c-84d9-b5be3eff11c5 clips=2 frames=930 queries=24 65s
[eval] stream 4/50 0475a0b9-9445-4083-8c6a-42f0171ff3e4 clips=1 frames=480 queries=5 27s
[eval] stream 5/50 04994ce8-9d47-44d6-a7b0-42b395e69390 clips=1 frames=1200 queries=30 111s
[eval] stream 6/50 055ba601-c243-4d4a-8a2c-694a5c88824c clips=1 frames=481 queries=12 27s
[eval] stream 7/50 056db3f1-f957-46c8-b16b-c8fce22e78f9 cli

In [706]:
import time, os, subprocess, glob, json

CHECK_INTERVAL_SEC = 180   # Local GPU hardware check every 3 minutes
DRIVE_CHECK_EVERY = 10     # Full checkpoint & training report every 30 minutes (10 * 3 min)
LOG_FILE = "/content/drive/MyDrive/selectstream_lite/logs/p4.log"
WORK_DIR = "/content/drive/MyDrive/selectstream_lite"

print(f"🟢 P4 Multi-Seed & Checkpoint Monitor started at {time.strftime('%X')}.")
print("• Local GPU checked every 3 min (zero Drive access).")
print("• Checkpoints, losses & milestones reported every 30 min.\n")

cycle = 0

while True:
    cycle += 1
    now = time.strftime('%X')

    # 1. Local GPU hardware check (zero Drive access)
    gpu_stat = subprocess.run(
        ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"],
        capture_output=True, text=True
    ).stdout.strip()

    # 2. Check if process is still running
    res = subprocess.run(["pgrep", "-f", "main.cli"], capture_output=True, text=True)
    is_running = bool(res.stdout.strip())

    print(f"[{now}] GPU: {gpu_stat} | P4 Active: {is_running}")

    # 3. Check Google Drive once every 30 minutes (Checkpoints + Loss + Evals)
    if cycle % DRIVE_CHECK_EVERY == 0:
        try:
            print(f"\n=======================================================")
            print(f"📋 [P4 Comprehensive Progress Report - {now}]")
            print(f"=======================================================")

            # A. Checkpoint Inspection (shows latest/ and final checkpoints)
            ckpt_dirs = glob.glob(f"{WORK_DIR}/checkpoints/W_*")
            print("💾 Checkpoint Status:")
            if ckpt_dirs:
                for cd in sorted(ckpt_dirs):
                    tag = os.path.basename(cd)
                    latest_pt = os.path.join(cd, "latest/trainable.pt")
                    final_pt = os.path.join(cd, "final/weights.pt")

                    if os.path.exists(final_pt):
                        mtime = time.strftime('%X', time.localtime(os.path.getmtime(final_pt)))
                        sz_mb = os.path.getsize(final_pt) / (1024 * 1024)
                        print(f"   ✅ {tag}: Final weights saved ({sz_mb:.1f} MB at {mtime})")
                    elif os.path.exists(latest_pt):
                        mtime = time.strftime('%X', time.localtime(os.path.getmtime(latest_pt)))
                        sz_mb = os.path.getsize(latest_pt) / (1024 * 1024)
                        print(f"   ⏳ {tag}: Training in progress (latest checkpoint saved at {mtime}, {sz_mb:.1f} MB)")
                    else:
                        print(f"   📁 {tag}: Initialized")
            else:
                print("   ⏳ Initializing first checkpoint folder...")

            # B. Parse latest training loss & step from p4.log
            if os.path.exists(LOG_FILE):
                tail = subprocess.run(["tail", "-n", "15", LOG_FILE], capture_output=True, text=True, timeout=10).stdout.strip()
                last_json_line = None
                for line in reversed(tail.splitlines()):
                    if line.strip().startswith("{") and "step" in line:
                        try:
                            last_json_line = json.loads(line)
                            break
                        except Exception:
                            pass

                if last_json_line:
                    st = last_json_line.get("step", "?")
                    pos = last_json_line.get("pos", "?")
                    loss = last_json_line.get("loss", 0.0)
                    loss_ans = last_json_line.get("loss_ans", 0.0)
                    loss_ret = last_json_line.get("loss_ret", 0.0)
                    recall = last_json_line.get("recall_at_m", 0.0)
                    print(f"\n📈 Live Training Step:")
                    print(f"   • Step: {st}/50 (Stream: {pos}/200)")
                    print(f"   • Loss Total: {loss:.3f} | Loss Answer: {loss_ans:.3f} | Loss Retrieval: {loss_ret:.3f}")
                    print(f"   • Recall@M: {recall:.1%}")
                else:
                    # If not in training, print latest eval status line
                    print(f"\n📁 Latest Log Line: {tail.splitlines()[-1] if tail else 'Waiting...'}")

            # C. Check finished evaluation metrics
            eval_summaries = glob.glob(f"{WORK_DIR}/eval/W_*_seed*/summary.json")
            if eval_summaries:
                print("\n🎯 Completed Evaluations:")
                for es in eval_summaries:
                    etag = os.path.basename(os.path.dirname(es))
                    try:
                        sdata = json.load(open(es))
                        acc = sdata.get("groups", {}).get("overall", {}).get("exact", {}).get("mean", "?")
                        cont = sdata.get("groups", {}).get("overall", {}).get("contains", {}).get("mean", "?")
                        print(f"   • {etag}: Contains={cont} | Exact={acc}")
                    except Exception:
                        print(f"   • {etag}: Finished evaluation")

            print("=======================================================\n")

        except Exception as e:
            # Fallback: Drive latency or timeout will NEVER crash the monitor
            print(f"  ⚠️ [Drive Notice]: Temporary Drive read issue ({e}). Skipping check without crashing.")

    # 4. Exit automatically when all P4 stages finish
    if not is_running:
        print(f"\n🎉 P4 Multi-Seed Pipeline Finished successfully at {time.strftime('%X')}!")
        break

    time.sleep(CHECK_INTERVAL_SEC)

🟢 P4 Multi-Seed & Checkpoint Monitor started at 05:19:40.
• Local GPU checked every 3 min (zero Drive access).
• Checkpoints, losses & milestones reported every 30 min.

[05:19:40] GPU: 0 %, 20432 MiB | P4 Active: True
[05:22:40] GPU: 0 %, 20432 MiB | P4 Active: True
[05:25:40] GPU: 0 %, 20432 MiB | P4 Active: True
[05:28:40] GPU: 39 %, 20432 MiB | P4 Active: True
[05:31:40] GPU: 0 %, 20432 MiB | P4 Active: True
[05:34:40] GPU: 0 %, 20432 MiB | P4 Active: True
[05:37:40] GPU: 14 %, 20432 MiB | P4 Active: True
[05:40:40] GPU: 14 %, 20432 MiB | P4 Active: True
[05:43:40] GPU: 0 %, 20432 MiB | P4 Active: True
[05:46:40] GPU: 0 %, 20432 MiB | P4 Active: True

📋 [P4 Comprehensive Progress Report - 05:46:40]
💾 Checkpoint Status:
   ✅ W_A_seed43: Final weights saved (1663.6 MB at 01:33:43)
   ✅ W_B_seed43: Final weights saved (1663.6 MB at 04:46:30)

📁 Latest Log Line: [eval] stream 42/50 25beabfc-48c0-4f81-b419-c47297aecf6f clips=1 frames=447 queries=13 37s

[05:49:40] GPU: 48 %, 20432 MiB |

--- 
## Step 5: Final Comparison Table (P2 on P3 + P4 Results)
Once P3 and P4 finish, run this cell to compile all predictions into a unified report with paired bootstrap comparisons.

In [707]:
import glob, os, sys, subprocess

env = dict(os.environ, PYTHONPATH=REPO_DIR)

# 1. Gather all predictions from Google Drive and Hub checkpoints
all_preds = glob.glob(os.path.join(WORK_DIR, "eval/**/predictions.jsonl"), recursive=True)
hub_preds = glob.glob("/content/ck/**/predictions.jsonl", recursive=True)

combined = {}
for p in hub_preds + all_preds:
    # Filter out empty files if any aborted early
    if os.path.exists(p) and os.path.getsize(p) > 0:
        tag = os.path.basename(os.path.dirname(p))
        combined[tag] = p

print(f"Discovered {len(combined)} total prediction runs:")
for tag, p in sorted(combined.items()):
    print(f"  • {tag}: {p}")

# 2. Select QA runs (including P3 Oracle and P4 Seed 43)
qa_runs = {
    k: v for k, v in combined.items()
    if ("qa" in k.lower() or "oracle" in k.lower()) and "uniform" not in k.lower() and "seed44" not in k.lower()
}

print(f"\nAnalyzing {len(qa_runs)} QA runs for P2 Report:")
qa_args = [f"{k}={v}" for k, v in sorted(qa_runs.items())]

# Reference run: prefer 'W_A_seed43_qa' or official 'eval_qa_A' / 'W_A_qa'
ref_candidate = "eval_qa_A"
if ref_candidate not in qa_runs:
    for candidate in ["W_A_seed43_qa", "qa_A", "W_A"]:
        matches = [k for k in qa_runs if candidate in k]
        if matches:
            ref_candidate = matches[0]
            break
    else:
        ref_candidate = list(qa_runs.keys())[0]

print(f"Reference run for paired bootstrap: {ref_candidate}")

# 3. Run P2 Analysis & Breakdowns (Runs on CPU in seconds)
FINAL_OUT = os.path.join(WORK_DIR, "eval/final_breakdown_all")
cmd_final = [
    sys.executable, "-m", "main.cli.eval_metrics_breakdown",
    "--preds", *qa_args,
    "--ref", ref_candidate,
    "--out_dir", FINAL_OUT
]
subprocess.run(cmd_final, env=env, check=True)

# 4. Display the Generated Summary
report_path = os.path.join(FINAL_OUT, "breakdown_summary.md")
if os.path.exists(report_path):
    print(f"\n🎉 Full breakdown report saved to: {report_path}\n")
    with open(report_path) as f:
        print(f.read())

Discovered 12 total prediction runs:
  • W_A_seed43_nlq: /content/drive/MyDrive/selectstream_lite/eval/W_A_seed43_nlq/predictions.jsonl
  • W_A_seed43_qa: /content/drive/MyDrive/selectstream_lite/eval/W_A_seed43_qa/predictions.jsonl
  • W_B_seed43_nlq: /content/drive/MyDrive/selectstream_lite/eval/W_B_seed43_nlq/predictions.jsonl
  • W_B_seed43_qa: /content/drive/MyDrive/selectstream_lite/eval/W_B_seed43_qa/predictions.jsonl
  • eval_nlq_A: /content/ck/runs/full_v3/eval_nlq_A/predictions.jsonl
  • eval_nlq_B: /content/ck/runs/full_v3/eval_nlq_B/predictions.jsonl
  • eval_nlq_C: /content/ck/runs/full_v3/eval_nlq_C/predictions.jsonl
  • eval_qa_A: /content/ck/runs/full_v3/eval_qa_A/predictions.jsonl
  • eval_qa_B: /content/ck/runs/full_v3/eval_qa_B/predictions.jsonl
  • eval_qa_C: /content/ck/runs/full_v3/eval_qa_C/predictions.jsonl
  • eval_qa_untrained: /content/ck/runs/full_v3/eval_qa_untrained/predictions.jsonl
  • p3_oracle_frames: /content/drive/MyDrive/selectstream_lite/eval/p3_or